# Clash Royale next-switch baseline

This notebook trains the unchanged expanded `72 → 128 → 1` switch model on the strict `same_collection` dataset. It runs three predetermined seeds for at most 60 epochs each, selects checkpoints by validation log loss, and never uses the test set for early stopping.

## One-time setup

1. Copy or sync the local `clash2` folder into the top level of Google Drive as `MyDrive/clash2`. The folder must include `data/switch_continuity_same_collection/arrays`; raw battle files are not needed.
2. In Google Drive, open `clash2/scripts/train_switch_baseline_colab.ipynb` with Google Colab.
3. In Colab, choose **Runtime → Change runtime type → T4 GPU**.
4. Run all cells. If the folder is elsewhere in Drive, change `PROJECT_FOLDER` below.

Progress and best checkpoints are written to Drive after every epoch. Re-running the notebook resumes interrupted seeds with the same settings.

In [ ]:
from pathlib import Path

PROJECT_FOLDER = Path("clash2")  # Relative to MyDrive.
MAX_EPOCHS = 60
PATIENCE = 8
MIN_DELTA = 1e-4
BATCH_SIZE = 512
SEEDS = [17, 3407, 918273]


In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")
PROJECT_ROOT = Path("/content/drive/MyDrive") / PROJECT_FOLDER
SOURCE_CACHE = PROJECT_ROOT / "data/switch_continuity_same_collection/arrays"
OUTPUT_DIR = PROJECT_ROOT / "data/switch_baseline_expanded_128"
required = [
    PROJECT_ROOT / "models/switch_model.py",
    PROJECT_ROOT / "scripts/train_switch_baseline.py",
    SOURCE_CACHE / "metadata.json",
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(missing))
os.chdir(PROJECT_ROOT)
print(f"Project: {PROJECT_ROOT}")
print(f"Persistent output: {OUTPUT_DIR}")


In [ ]:
import shutil
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Select Runtime → Change runtime type → T4 GPU.")
print(f"GPU: {torch.cuda.get_device_name(0)}")

# Drive is durable but slower for frequent reads. Copy the 82 MB prepared cache
# into this Colab session; checkpoints still go directly to Drive.
SESSION_CACHE = Path("/content/clash2_same_collection_arrays")
source_metadata = (SOURCE_CACHE / "metadata.json").read_bytes()
cache_is_current = (
    (SESSION_CACHE / "metadata.json").exists()
    and (SESSION_CACHE / "metadata.json").read_bytes() == source_metadata
)
if not cache_is_current:
    if SESSION_CACHE.exists():
        shutil.rmtree(SESSION_CACHE)
    shutil.copytree(SOURCE_CACHE, SESSION_CACHE)
print(f"Session cache ready: {SESSION_CACHE}")


## Train or resume

The architecture, optimizer, learning rate, weight decay, batch size, and strict player split remain fixed. Early stopping waits for eight consecutive epochs without at least `0.0001` improvement in validation log loss. A run can reach all 60 epochs when validation continues to improve.

In [ ]:
import subprocess
import sys

command = [
    sys.executable, "-u", str(PROJECT_ROOT / "scripts/train_switch_baseline.py"),
    "--cache", str(SESSION_CACHE),
    "--output-dir", str(OUTPUT_DIR),
    "--device", "cuda",
    "--max-epochs", str(MAX_EPOCHS),
    "--patience", str(PATIENCE),
    "--min-delta", str(MIN_DELTA),
    "--batch-size", str(BATCH_SIZE),
    "--seeds", *map(str, SEEDS),
]
print("Running:", " ".join(command))
subprocess.run(command, check=True)


## Review the baseline

This table shows every seed, followed by the mean and sample standard deviation. The plot makes convergence and early stopping easy to inspect.

In [ ]:
import json
import pandas as pd
from IPython.display import display

results_path = OUTPUT_DIR / "results.json"
report = json.loads(results_path.read_text(encoding="utf-8"))
rows = []
for run in report["runs"]:
    rows.append({
        "seed": run["seed"],
        "best_epoch": run["best_epoch"],
        "epochs_completed": run["epochs_completed"],
        **{f"validation_{key}": value for key, value in run["validation"].items()},
        **{f"test_{key}": value for key, value in run["test"].items()},
    })
display(pd.DataFrame(rows).set_index("seed").round(6))
print("\nAggregate test metrics (mean ± sample standard deviation)")
for name, values in report["aggregate"]["test"].items():
    print(f"{name:18s} {values['mean']:.6f} ± {values['sample_standard_deviation']:.6f}")


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for run in report["runs"]:
    epochs = [row["epoch"] for row in run["history"]]
    axes[0].plot(epochs, [row["train_loss"] for row in run["history"]], label=f"seed {run['seed']}")
    axes[1].plot(epochs, [row["validation_log_loss"] for row in run["history"]], label=f"seed {run['seed']}")
    axes[1].axvline(run["best_epoch"], linestyle=":", alpha=0.35)
axes[0].set(title="Training loss", xlabel="Epoch", ylabel="Binary cross-entropy")
axes[1].set(title="Validation log loss", xlabel="Epoch", ylabel="Log loss")
for axis in axes:
    axis.grid(alpha=0.25)
    axis.legend()
plt.tight_layout()
plt.show()


## Outputs

All outputs are in `data/switch_baseline_expanded_128` in Drive. `results.json` contains the full audit record and aggregate statistics. Each seed has a portable best checkpoint, an epoch-by-epoch progress checkpoint, and an individual result file. These generated files remain ignored by Git.